# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Rule: A page needs review if it's stale-but-visible, declining-with-demand,
thin-but-visible, slipping-on-page-1, or getting impressions but not clicks.

Reason codes:
- stale_visible_page: not updated in 180+ days AND 500+ impressions
- declining_with_demand: trend_direction == "down" AND 100+ impressions  
- thin_visible_page: word_count < 1200 AND 250+ impressions
- page_one_decay_risk: avg_position <= 10 AND page is 180+ days old
- low_ctr_visible_page: 500+ impressions, position 1-20, but ctr < 0.5
- general_refresh_review: fallback when none of the above fire

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# find repo root by walking up until we find the data folder
p = Path.cwd()
while not (p / "data" / "raw" / "content_refresh_anonymized.csv").exists():
    p = p.parent
ROOT = p

df = pd.read_csv(ROOT / "data" / "raw" / "content_refresh_anonymized.csv")

def reason_codes(row):
    reasons = []
    if row["days_since_last_update"] >= 180 and row["impressions_90d"] >= 500:
        reasons.append("stale_visible_page")
    if row["trend_direction"].lower() == "down" and row["impressions_90d"] >= 100:
        reasons.append("declining_with_demand")
    if row["word_count"] > 0 and row["word_count"] < 1200 and row["impressions_90d"] >= 250:
        reasons.append("thin_visible_page")
    if row["avg_position"] > 0 and row["avg_position"] <= 10 and row["content_age_days"] >= 180:
        reasons.append("page_one_decay_risk")
    if row["impressions_90d"] >= 500 and 0 < row["avg_position"] <= 20 and row["ctr"] < 0.5:
        reasons.append("low_ctr_visible_page")
    if not reasons:
        reasons.append("general_refresh_review")
    return "|".join(reasons)

df["reason_codes"] = df.apply(reason_codes, axis=1)

# score: normalize impressions (visibility) + staleness, weight them
def pct_rank(s):
    return s.rank(pct=True)

df["visibility_score"] = pct_rank(np.log1p(df["impressions_90d"]))
df["freshness_risk_score"] = pct_rank(df["days_since_last_update"])
df["baseline_score"] = (0.5 * df["visibility_score"] + 0.5 * df["freshness_risk_score"]).clip(0, 1)

df["rank"] = df["baseline_score"].rank(method="first", ascending=False).astype(int)
out = df.sort_values("rank")[["content_id", "client_id", "rank", "baseline_score",
                                "reason_codes", "impressions_90d", "days_since_last_update",
                                "avg_position", "ctr", "trend_direction", "is_declining_label"
                                if "is_declining_label" in df.columns else "trend_direction"]]

out_path = ROOT / "work" / "outputs" / "baseline_action_score.csv"
out_path.parent.mkdir(parents=True, exist_ok=True)
out.to_csv(out_path, index=False)
print(f"Wrote {len(out)} rows to {out_path}")
out.head(20)

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
top20 = out.head(20)
top20

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
used_columns = ["impressions_90d", "days_since_last_update"]  # whatever you scored on
leaky = [c for c in used_columns if c in ("trend_pct", "health_score", "priority_score", "action_type")]
print("Leaky columns used:", leaky)  # should print []

## Self-check

Before you submit, confirm each line honestly:

- ✅ Every section above is filled — markdown thinking AND the code that backs it
- ✅ The notebook runs top to bottom with no errors (Runtime → Run all)
- ✅ No client names, URLs, or private queries anywhere
- ✅ My claims use careful words: observed, measured, directional, decision-support
- ✅ Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.